Pilot on the v2 ablation split (never main data): checks the batch path and prompt 2's answer format.

Criterion fixed before running: invalid rate < 10% per provider and task. If exceeded, prompt 2 is fixed once and the pilot re-run;
verification prompt stays locked. OpenAI/Anthropic via batch (results within 24 h), Google live.
Needs the pilot renders from `render_main.ipynb` first. Cost ~$3-5.

In [1]:
import json

import duckdb
import pandas as pd

from thesis.config import REPO_ROOT, load_config
from thesis.vlm.run import configured_clients, run_dataset

out_dir = REPO_ROOT / "data" / "dataset_v2"
pilot_dir = out_dir / "pilot"
pilot_dir.mkdir(exist_ok=True)
INVALID_LIMIT = 0.10

prompts = load_config("prompts")
TASK_PROMPTS = {
    "verification": next(p["text"] for p in prompts["prompts"] if p["id"] == "roleplaying"),
    "distance": next(p["text"] for p in prompts["distance_prompts"] if p["id"] == "roleplaying"),
}
clients = {client.provider: client for client in configured_clients()}

selected = pd.read_parquet(out_dir / "selection" / "sources.parquet")
pins = pd.read_parquet(out_dir / "audit" / "selected_pins.parquet")
gold = selected.merge(pd.read_json(out_dir / "gold" / "is_building.jsonl", lines=True), on="source_id")
ablation_ids = gold.loc[gold["is_building"] & (gold["assignment"] == "ablation"), "source_id"]
pilot_pins = pins.loc[
    pins["source_id"].isin(ablation_ids) & pins["bin_id"].isin(load_config("scoring")["primary_bins"])
]
with duckdb.connect(str(out_dir / "manifest.duckdb"), read_only=True) as connection:
    renders = connection.execute("SELECT pin_id, image_path FROM renders WHERE preset_id = 'google_pin_red'").fetch_df()
pilot_pins = pilot_pins.merge(renders, on="pin_id")
images = [(out_dir / path).read_bytes() for path in pilot_pins["image_path"]]
len(pilot_pins), pilot_pins["source_id"].nunique()

(713, 92)

In [2]:
# submit once: OpenAI + Anthropic batches, Google live (blocking, ~1-2k calls)
batch_ids = {
    provider: {
        task: clients[provider].batch_classify([(image, prompt) for image in images], task=task)
        for task, prompt in TASK_PROMPTS.items()
    }
    for provider in ("openai", "anthropic")
}
(pilot_dir / "batches.json").write_text(json.dumps(batch_ids, indent=2))
for task, prompt in TASK_PROMPTS.items():
    run_dataset([(image, prompt) for image in images], task=task, clients=[clients["google"]])
batch_ids

{'openai': {'verification': ['batch_6ac3ba20c11881909a178ccee8168dc7',
   'batch_6ac3ba7710b48190b8ffc7bb55a87a2c',
   'batch_6ac3ba8bbd7c81909f9bfd3a8d484530'],
  'distance': ['batch_6ac3badf829c8190b1fc6fa1c823dc57',
   'batch_6ac3bb367470819095c9ebc8b9720f13',
   'batch_6ac3bb4aa3d48190aee06bd0be8e8c3c']},
 'anthropic': {'verification': ['msgbatch_01DwKa6xXXBbzBcXCook3Pwv',
   'msgbatch_01AwTAExNSnPoPNdfwkng392'],
  'distance': ['msgbatch_01WmQyWLYF9LRBFBbbeZtUmB',
   'msgbatch_01CjEXusnWsnxEzTShXyCyhe']}}

In [10]:
# re-run until every batch is collected (None = still running)
batch_ids = json.loads((pilot_dir / "batches.json").read_text())
{
    (provider, task, batch_id): clients[provider].collect_batch(batch_id)
    for provider, tasks in batch_ids.items()
    for task, ids in tasks.items()
    for batch_id in ids
}

{('openai', 'verification', 'batch_6ac3ba20c11881909a178ccee8168dc7'): 319,
 ('openai', 'verification', 'batch_6ac3ba7710b48190b8ffc7bb55a87a2c'): 311,
 ('openai', 'verification', 'batch_6ac3ba8bbd7c81909f9bfd3a8d484530'): 83,
 ('openai', 'distance', 'batch_6ac3badf829c8190b1fc6fa1c823dc57'): 319,
 ('openai', 'distance', 'batch_6ac3bb367470819095c9ebc8b9720f13'): 311,
 ('openai', 'distance', 'batch_6ac3bb4aa3d48190aee06bd0be8e8c3c'): 83,
 ('anthropic', 'verification', 'msgbatch_01DwKa6xXXBbzBcXCook3Pwv'): 397,
 ('anthropic', 'verification', 'msgbatch_01AwTAExNSnPoPNdfwkng392'): 316,
 ('anthropic', 'distance', 'msgbatch_01WmQyWLYF9LRBFBbbeZtUmB'): 396,
 ('anthropic', 'distance', 'msgbatch_01CjEXusnWsnxEzTShXyCyhe'): 317}

In [11]:
# read everything from the cache (anything a batch failed on is sent live here), then check the criterion
rows = []
for task, prompt in TASK_PROMPTS.items():
    for provider, responses in run_dataset(
        [(image, prompt) for image in images], task=task, clients=list(clients.values())
    ).items():
        for pin, response in zip(pilot_pins.itertuples(), responses):
            r = response.vlm_response
            rows.append(
                {
                    "task": task,
                    "provider": provider,
                    "pin_id": pin.pin_id,
                    "bin_id": pin.bin_id,
                    "parsed_answer": r.parsed_answer,
                    "invalid": r.refusal_error_state is not None,
                    "empty": r.raw_text.strip() == "",
                    "tier": r.price_schedule["tier"],
                    "cost": r.cost,
                }
            )
pilot = pd.DataFrame(rows)
pilot.to_parquet(pilot_dir / "answers.parquet")
summary = pilot.groupby(["task", "provider"]).agg(
    n=("invalid", "size"),
    invalid=("invalid", "mean"),
    empty=("empty", "mean"),
    batch_share=("tier", lambda t: (t == "batch").mean()),
    cost=("cost", "sum"),
)
summary["passes"] = summary["invalid"] < INVALID_LIMIT
summary

n   invalid     empty  batch_share      cost  passes
task         provider                                                         
distance     anthropic  713  0.001403  0.000000          1.0  2.070387    True
             google     713  0.000000  0.000000          0.0  2.540659    True
             openai     713  0.000000  0.000000          1.0  2.586394    True
verification anthropic  713  0.009818  0.008415          1.0  1.508282    True
             google     713  0.000000  0.000000          0.0  1.038103    True
             openai     713  0.000000  0.000000          1.0  0.805934    True